# Notebook 3: Train / Validation / Test Split
**MLOps Training 2026/2027 · Task 2: From Tables to Notebooks**

### Objective:
1. Load the labeled dataset from Notebook 2 (`labeled_orders.parquet`).
2. Partition the dataset **before performing any deep exploratory analysis or feature engineering** to prevent test set data leakage.
3. Analyze and compare splitting strategies (**Temporal/Time-based vs. Stratified Random**) by evaluating date ranges, monthly order volume, and class balance stability.
4. Implement a **70% Train / 15% Validation / 15% Test Stratified Random Split** (`random_state=42`) that strictly preserves the 8.11% positive class ratio across all partitions.
5. Verify partition integrity (zero overlap, complete coverage, identical target distributions).
6. Export the split artifacts (`train.parquet`, `val.parquet`, `test.parquet`) for Notebook 4 (EDA), Notebook 5 (Feature Engineering), and Notebook 6 (Modeling).

In [1]:
import os
from pathlib import Path
import pandas as pd
import numpy as np
from dotenv import load_dotenv
from sklearn.model_selection import StratifiedShuffleSplit
from IPython.display import display

# Display configuration
pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 1000)

print('Libraries imported successfully.')

Libraries imported successfully.


In [2]:
# Locate repository root and artifacts directories
def find_project_root(start_path: Path) -> Path:
    """Traverse up directory tree to locate repository root containing .env or .git."""
    for parent in [start_path] + list(start_path.parents):
        if (parent / '.env').exists() or (parent / '.git').exists():
            return parent
    return start_path

root_dir = find_project_root(Path.cwd())
load_dotenv(dotenv_path=root_dir / '.env')

# Input artifact from Notebook 2
input_artifact_path = root_dir / 'tasks' / 'task-02-notebooks' / 'artifacts' / '02_create_labels' / 'labeled_orders.parquet'

# Output artifact directory for Notebook 3
output_artifact_dir = root_dir / 'tasks' / 'task-02-notebooks' / 'artifacts' / '03_split'
output_artifact_dir.mkdir(parents=True, exist_ok=True)

print(f'Project root           : {root_dir}')
print(f'Input artifact path    : {input_artifact_path}')
print(f'Output artifact dir    : {output_artifact_dir}')

Project root           : /Users/halakhalifa/Desktop/qafza/olist-mlops
Input artifact path    : /Users/halakhalifa/Desktop/qafza/olist-mlops/tasks/task-02-notebooks/artifacts/02_create_labels/labeled_orders.parquet
Output artifact dir    : /Users/halakhalifa/Desktop/qafza/olist-mlops/tasks/task-02-notebooks/artifacts/03_split


## Step 1: Load Labeled Dataset

We load `labeled_orders.parquet` produced in Notebook 2.

In [3]:
assert input_artifact_path.exists(), f'Input artifact not found at {input_artifact_path}! Run Notebook 2 first.'
df_labeled = pd.read_parquet(input_artifact_path)

print(f'Loaded labeled dataset: {df_labeled.shape[0]:,} rows x {df_labeled.shape[1]} columns')
print(f'Target variable value counts:')
print(df_labeled['is_late'].value_counts())
display(df_labeled[['order_id', 'order_purchase_timestamp', 'is_late']].head(3))

Loaded labeled dataset: 96,470 rows x 37 columns
Target variable value counts:
is_late
0    88644
1     7826
Name: count, dtype: int64


,order_id,order_purchase_timestamp,is_late
0,e481f51cbdc54678b7cc49136f2d6af7,2017-10-02 10:56:33,0
1,53cdb2fc8bc7dce0b6741e2150273451,2018-07-24 20:41:37,0
2,47770eb9100c2d0c44946d9cf07ec65d,2018-08-08 08:38:49,0


## Step 2: Split Strategy Analysis (Temporal vs. Stratified Random)

Before partitioning, we evaluate the properties of the dataset to decide between **Temporal (Time-based)** and **Stratified Random** splitting:

1. **Date Range & Volume Stability**: Check the span of `order_purchase_timestamp` and monthly order counts.
2. **Temporal Drift in Late Rate**: Check whether the late delivery rate fluctuates across months (due to macro events such as the Brazilian courier/trucker strikes in early 2018 or holiday volume peaks in Nov 2017).
3. **Comparison**: Evaluate label balance and partition characteristics under both approaches.

In [4]:
# 1. Date range examination
min_date = df_labeled['order_purchase_timestamp'].min()
max_date = df_labeled['order_purchase_timestamp'].max()
timespan_days = (max_date - min_date).days

print(f'Order Purchase Date Range: {min_date.strftime("%Y-%m-%d %H:%M")} to {max_date.strftime("%Y-%m-%d %H:%M")} ({timespan_days} days)')

# 2. Monthly order volume and late rate stability
df_temp = df_labeled.copy()
df_temp['year_month'] = df_temp['order_purchase_timestamp'].dt.to_period('M')
monthly_summary = df_temp.groupby('year_month').agg(
    total_orders=('order_id', 'count'),
    late_orders=('is_late', 'sum'),
    late_rate=('is_late', 'mean')
)
monthly_summary['pct_of_all_orders'] = (monthly_summary['total_orders'] / len(df_labeled)) * 100
monthly_summary['late_rate_pct'] = monthly_summary['late_rate'] * 100

print('\n=== Monthly Breakdown (Order Volume & Late Rate) ===')
display(monthly_summary[['total_orders', 'pct_of_all_orders', 'late_orders', 'late_rate_pct']])

Order Purchase Date Range: 2016-09-15 12:16 to 2018-08-29 15:00 (713 days)

=== Monthly Breakdown (Order Volume & Late Rate) ===


,total_orders,pct_of_all_orders,late_orders,late_rate_pct
year_month,,,,
2016-09,1,0.001037,1,100.000000
2016-10,265,0.274697,3,1.132075
2016-12,1,0.001037,0,0.000000
2017-01,750,0.777444,23,3.066667
2017-02,1653,1.713486,53,3.206292
2017-03,2546,2.639162,142,5.577376
2017-04,2303,2.387271,181,7.859314
2017-05,3545,3.674718,128,3.610719
2017-06,3135,3.249715,121,3.859649


In [5]:
# 3. Simulate and compare both strategies

# --- Strategy A: Temporal Split (First 70% Train, Next 15% Val, Last 15% Test) ---
df_sorted = df_labeled.sort_values('order_purchase_timestamp').reset_index(drop=True)
n = len(df_sorted)
n_train = int(0.70 * n)
n_val = int(0.15 * n)

t_train = df_sorted.iloc[:n_train]
t_val = df_sorted.iloc[n_train:n_train + n_val]
t_test = df_sorted.iloc[n_train + n_val:]

# --- Strategy B: Stratified Random Split (70 / 15 / 15) ---
sss1 = StratifiedShuffleSplit(n_splits=1, test_size=0.30, random_state=42)
train_idx, temp_idx = next(sss1.split(df_labeled, df_labeled['is_late']))
r_train = df_labeled.iloc[train_idx]
r_temp = df_labeled.iloc[temp_idx]

sss2 = StratifiedShuffleSplit(n_splits=1, test_size=0.50, random_state=42)
val_idx, test_idx = next(sss2.split(r_temp, r_temp['is_late']))
r_val = r_temp.iloc[val_idx]
r_test = r_temp.iloc[test_idx]

comparison_df = pd.DataFrame([
    {
        'Strategy': 'Temporal (Time-based)',
        'Split': 'Train (70%)',
        'Rows': f'{len(t_train):,}',
        'Date Range': f'{t_train["order_purchase_timestamp"].min().strftime("%Y-%m-%d")} to {t_train["order_purchase_timestamp"].max().strftime("%Y-%m-%d")}',
        'Late Count': f'{t_train["is_late"].sum():,}',
        'Late Rate (%)': f'{t_train["is_late"].mean()*100:.2f}%'
    },
    {
        'Strategy': 'Temporal (Time-based)',
        'Split': 'Validation (15%)',
        'Rows': f'{len(t_val):,}',
        'Date Range': f'{t_val["order_purchase_timestamp"].min().strftime("%Y-%m-%d")} to {t_val["order_purchase_timestamp"].max().strftime("%Y-%m-%d")}',
        'Late Count': f'{t_val["is_late"].sum():,}',
        'Late Rate (%)': f'{t_val["is_late"].mean()*100:.2f}%'
    },
    {
        'Strategy': 'Temporal (Time-based)',
        'Split': 'Test (15%)',
        'Rows': f'{len(t_test):,}',
        'Date Range': f'{t_test["order_purchase_timestamp"].min().strftime("%Y-%m-%d")} to {t_test["order_purchase_timestamp"].max().strftime("%Y-%m-%d")}',
        'Late Count': f'{t_test["is_late"].sum():,}',
        'Late Rate (%)': f'{t_test["is_late"].mean()*100:.2f}%'
    },
    {
        'Strategy': 'Stratified Random (70/15/15)',
        'Split': 'Train (70%)',
        'Rows': f'{len(r_train):,}',
        'Date Range': f'{r_train["order_purchase_timestamp"].min().strftime("%Y-%m-%d")} to {r_train["order_purchase_timestamp"].max().strftime("%Y-%m-%d")}',
        'Late Count': f'{r_train["is_late"].sum():,}',
        'Late Rate (%)': f'{r_train["is_late"].mean()*100:.2f}%'
    },
    {
        'Strategy': 'Stratified Random (70/15/15)',
        'Split': 'Validation (15%)',
        'Rows': f'{len(r_val):,}',
        'Date Range': f'{r_val["order_purchase_timestamp"].min().strftime("%Y-%m-%d")} to {r_val["order_purchase_timestamp"].max().strftime("%Y-%m-%d")}',
        'Late Count': f'{r_val["is_late"].sum():,}',
        'Late Rate (%)': f'{r_val["is_late"].mean()*100:.2f}%'
    },
    {
        'Strategy': 'Stratified Random (70/15/15)',
        'Split': 'Test (15%)',
        'Rows': f'{len(r_test):,}',
        'Date Range': f'{r_test["order_purchase_timestamp"].min().strftime("%Y-%m-%d")} to {r_test["order_purchase_timestamp"].max().strftime("%Y-%m-%d")}',
        'Late Count': f'{r_test["is_late"].sum():,}',
        'Late Rate (%)': f'{r_test["is_late"].mean()*100:.2f}%'
    }
])

print('=== Comparison of Splitting Strategies ===')
display(comparison_df)

=== Comparison of Splitting Strategies ===


,Strategy,Split,Rows,Date Range,Late Count,Late Rate (%)
0,Temporal (Time-based),Train (70%),"67,529",2016-09-15 to 2018-04-15,"6,096",9.03%
1,Temporal (Time-based),Validation (15%),"14,470",2018-04-15 to 2018-06-21,773,5.34%
2,Temporal (Time-based),Test (15%),"14,471",2018-06-21 to 2018-08-29,957,6.61%
3,Stratified Random (70/15/15),Train (70%),"67,529",2016-10-03 to 2018-08-29,"5,478",8.11%
4,Stratified Random (70/15/15),Validation (15%),"14,470",2016-09-15 to 2018-08-28,"1,174",8.11%
5,Stratified Random (70/15/15),Test (15%),"14,471",2016-10-03 to 2018-08-28,"1,174",8.11%


### Split Strategy Decision & Justification:

- **Temporal Split Evaluation**:
  - In this dataset, the late delivery rate varies heavily across chronological windows due to real-world operational events (e.g., Brazilian nationwide logistics/courier strikes in March 2018 resulted in a **21.36%** late rate, whereas June 2018 dropped to **1.36%**).
  - Consequently, a strict temporal split yields a significant distribution shift (Train: 9.03%, Val: 5.34%, Test: 6.61%), causing high evaluation variance.

- **Stratified Random Split Selection**:
  - A **Stratified Random Split (70% Train / 15% Val / 15% Test with `random_state=42`)** guarantees exact class representation (**8.11%** in all three partitions).
  - It exposes the training, validation, and test sets to all operational regimes, seasonal cycles, and product categories uniformly, ensuring robust and reproducible evaluation metrics.

## Step 3: Implement Stratified Random Split

We partition the dataset into `train_df` (70%), `val_df` (15%), and `test_df` (15%).

In [6]:
# 1. First split: 70% Train, 30% Temporary
sss1 = StratifiedShuffleSplit(n_splits=1, test_size=0.30, random_state=42)
train_idx, temp_idx = next(sss1.split(df_labeled, df_labeled['is_late']))

train_df = df_labeled.iloc[train_idx].copy().reset_index(drop=True)
temp_df = df_labeled.iloc[temp_idx].copy().reset_index(drop=True)

# 2. Second split: 15% Validation (50% of Temp), 15% Test (50% of Temp)
sss2 = StratifiedShuffleSplit(n_splits=1, test_size=0.50, random_state=42)
val_idx, test_idx = next(sss2.split(temp_df, temp_df['is_late']))

val_df = temp_df.iloc[val_idx].copy().reset_index(drop=True)
test_df = temp_df.iloc[test_idx].copy().reset_index(drop=True)

print(f'Train set shape      : {train_df.shape} (70.0%)')
print(f'Validation set shape : {val_df.shape} (15.0%)')
print(f'Test set shape       : {test_df.shape} (15.0%)')

Train set shape      : (67529, 37) (70.0%)
Validation set shape : (14470, 37) (15.0%)
Test set shape       : (14471, 37) (15.0%)


## Step 4: Verification & Integrity Checks

We verify:
1. Mutual exclusivity (zero overlap of `order_id`s across splits).
2. Complete union ($N_{\text{train}} + N_{\text{val}} + N_{\text{test}} = N_{\text{total}}$).
3. Target class balance consistency across all three splits.
4. Zero missing values in `is_late`.

In [7]:
# 1. Verify mutual exclusivity (zero ID overlap)
train_orders = set(train_df['order_id'])
val_orders = set(val_df['order_id'])
test_orders = set(test_df['order_id'])

assert len(train_orders.intersection(val_orders)) == 0, 'Overlap detected between Train and Validation!'
assert len(train_orders.intersection(test_orders)) == 0, 'Overlap detected between Train and Test!'
assert len(val_orders.intersection(test_orders)) == 0, 'Overlap detected between Validation and Test!'
print('✓ Mutual exclusivity check passed: Zero order overlap between splits.')

# 2. Verify complete union
total_split_rows = len(train_df) + len(val_df) + len(test_df)
assert total_split_rows == len(df_labeled), f'Row count mismatch! Expected {len(df_labeled)}, got {total_split_rows}'
print(f'✓ Union completeness check passed: Exactly {total_split_rows:,} orders accounted for.')

# 3. Verify target class proportions
splits_summary = pd.DataFrame([
    {
        'Split': 'Train',
        'Total Rows': len(train_df),
        'Pct of Total (%)': len(train_df) / len(df_labeled) * 100,
        'On-Time (0)': (train_df['is_late'] == 0).sum(),
        'Late (1)': (train_df['is_late'] == 1).sum(),
        'Late Rate (%)': train_df['is_late'].mean() * 100
    },
    {
        'Split': 'Validation',
        'Total Rows': len(val_df),
        'Pct of Total (%)': len(val_df) / len(df_labeled) * 100,
        'On-Time (0)': (val_df['is_late'] == 0).sum(),
        'Late (1)': (val_df['is_late'] == 1).sum(),
        'Late Rate (%)': val_df['is_late'].mean() * 100
    },
    {
        'Split': 'Test',
        'Total Rows': len(test_df),
        'Pct of Total (%)': len(test_df) / len(df_labeled) * 100,
        'On-Time (0)': (test_df['is_late'] == 0).sum(),
        'Late (1)': (test_df['is_late'] == 1).sum(),
        'Late Rate (%)': test_df['is_late'].mean() * 100
    }
])

print('\n=== Final Split Summary ===')
display(splits_summary)

# Assert late rate is identical within 0.01%
assert abs(train_df['is_late'].mean() - df_labeled['is_late'].mean()) < 1e-4
assert abs(val_df['is_late'].mean() - df_labeled['is_late'].mean()) < 1e-4
assert abs(test_df['is_late'].mean() - df_labeled['is_late'].mean()) < 1e-4
print('✓ Class balance check passed: Exactly 8.11% late rate preserved across all splits.')

✓ Mutual exclusivity check passed: Zero order overlap between splits.
✓ Union completeness check passed: Exactly 96,470 orders accounted for.

=== Final Split Summary ===


,Split,Total Rows,Pct of Total (%),On-Time (0),Late (1),Late Rate (%)
0,Train,67529,70.000000,62051,5478,8.112070
1,Validation,14470,14.999482,13296,1174,8.113338
2,Test,14471,15.000518,13297,1174,8.112777


✓ Class balance check passed: Exactly 8.11% late rate preserved across all splits.


## Step 5: Save Artifacts for Downstream Notebooks

We export:
- `train.parquet` &rarr; Used for exploratory data analysis (Notebook 4) and feature engineering / fitting transformers (Notebook 5).
- `val.parquet` &rarr; Used for hyperparameter tuning and model selection (Notebook 6).
- `test.parquet` &rarr; Held out strictly for final unbiased model evaluation (Notebook 6).

In [8]:
train_path = output_artifact_dir / 'train.parquet'
val_path = output_artifact_dir / 'val.parquet'
test_path = output_artifact_dir / 'test.parquet'

train_df.to_parquet(train_path, index=False)
val_df.to_parquet(val_path, index=False)
test_df.to_parquet(test_path, index=False)

print(f'✓ Saved Train artifact      : {train_path} ({train_path.stat().st_size / (1024*1024):.2f} MB, {len(train_df):,} rows)')
print(f'✓ Saved Validation artifact : {val_path} ({val_path.stat().st_size / (1024*1024):.2f} MB, {len(val_df):,} rows)')
print(f'✓ Saved Test artifact       : {test_path} ({test_path.stat().st_size / (1024*1024):.2f} MB, {len(test_df):,} rows)')

print('\n=======================================================')
print('✓ Notebook 3 execution completed successfully!')
print('✓ Dataset splits are ready for Notebook 4 (EDA) and Notebook 5 (Feature Engineering).')
print('=======================================================')

✓ Saved Train artifact      : /Users/halakhalifa/Desktop/qafza/olist-mlops/tasks/task-02-notebooks/artifacts/03_split/train.parquet (13.15 MB, 67,529 rows)
✓ Saved Validation artifact : /Users/halakhalifa/Desktop/qafza/olist-mlops/tasks/task-02-notebooks/artifacts/03_split/val.parquet (3.08 MB, 14,470 rows)
✓ Saved Test artifact       : /Users/halakhalifa/Desktop/qafza/olist-mlops/tasks/task-02-notebooks/artifacts/03_split/test.parquet (3.07 MB, 14,471 rows)

✓ Notebook 3 execution completed successfully!
✓ Dataset splits are ready for Notebook 4 (EDA) and Notebook 5 (Feature Engineering).
